# Hotel Management AI Agent

In [22]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio
import ollama
import sqlite3 
import re
from datetime import date
from openai import OpenAI


In [23]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI(api_key=openai_api_key)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(base_url=ollama_url, api_key="ollama")

MODEL = "openai/gpt-oss-120b"
DB = "hotel.db"

# Database

In [24]:
def create_customers_table():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS customers (
                customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
                full_name TEXT NOT NULL,
                phone TEXT NOT NULL,
                email TEXT,
                room_number TEXT NOT NULL,
                check_in TEXT NOT NULL,
                check_out TEXT NOT NULL,
                special_requests TEXT
            )
        """)
        conn.commit()

create_customers_table()

In [25]:
# check the table is exists
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    print(cursor.fetchall())

[('customers',), ('sqlite_sequence',)]


# Tool functions (validation, save, view, update, delete)

Validation (validate the customer data)

In [26]:
def validate_customer_data(full_name=None, phone=None, email=None,
                            room_number=None, check_in=None, check_out=None):
    """Returns (True, None) if valid, or (False, 'reason') if not.
    Only checks fields that are provided (not None)."""

    if full_name is not None and full_name.strip() == "":
        return False, "Full name cannot be empty."

    if phone is not None:
        if not re.fullmatch(r"\+?\d{9,15}", phone):
            return False, "Phone number must be 9 to 15 digits, optionally starting with +."

    if email is not None and email.strip() != "":
        if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email):
            return False, "Email address doesn't look valid."

    if room_number is not None and room_number.strip() == "":
        return False, "Room number cannot be empty."

    if check_in is not None:
        try:
            date.fromisoformat(check_in)
        except ValueError:
            return False, "Check-in date must be in YYYY-MM-DD format."

    if check_out is not None:
        try:
            date.fromisoformat(check_out)
        except ValueError:
            return False, "Check-out date must be in YYYY-MM-DD format."

    if check_in is not None and check_out is not None:
        if date.fromisoformat(check_out) <= date.fromisoformat(check_in):
            return False, "Check-out date must be after the check-in date."

    return True, None

Save customer data

In [27]:
def save_customer(full_name, phone, room_number, check_in, check_out,
                   email=None, special_requests=None):
    valid, error = validate_customer_data(
        full_name=full_name, phone=phone, email=email,
        room_number=room_number, check_in=check_in, check_out=check_out
    )
    if not valid:
        return {"success": False, "error": error}

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO customers (full_name, phone, email, room_number,
                                    check_in, check_out, special_requests)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (full_name, phone, email, room_number, check_in, check_out, special_requests))
        conn.commit()
        new_id = cursor.lastrowid

    return {"success": True, "customer_id": new_id}

In [7]:
# Save a customer to test the function
print(save_customer("Amara Perera", "+94712223344", "A12", "2026-11-01", "2026-11-03", email="amara@example.com"))

{'success': True, 'customer_id': 1}


In [28]:
# Invalid input, rejected
print(save_customer("Bad Phone", "123", "101", "2026-10-01", "2026-10-04"))
# Expect: success False, error about phone digits

print(save_customer("Bad Dates", "0712345678", "101", "2026-10-05", "2026-10-01"))
# Expect: success False, error about check-out after check-in

{'success': False, 'error': 'Phone number must be 9 to 15 digits, optionally starting with +.'}
{'success': False, 'error': 'Check-out date must be after the check-in date.'}


View (Search customer data)

In [29]:
def view_customer(customer_id=None, name=None):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        if customer_id is not None:
            cursor.execute("SELECT * FROM customers WHERE customer_id = ?", (customer_id,))
        elif name is not None:
            cursor.execute("SELECT * FROM customers WHERE full_name LIKE ?", (f"%{name}%",))
        else:
            return {"success": False, "error": "Provide a customer_id or a name to search."}

        columns = [d[0] for d in cursor.description]
        rows = cursor.fetchall()

    if not rows:
        return {"success": False, "error": "No matching customer found."}

    results = [dict(zip(columns, row)) for row in rows]

    if len(results) > 1:
        return {"success": True, "multiple_matches": True, "results": results}

    return {"success": True, "multiple_matches": False, "result": results[0]}

In [30]:
# view by id
print(view_customer(customer_id=1))

{'success': True, 'multiple_matches': False, 'result': {'customer_id': 1, 'full_name': 'Amara Perera', 'phone': '+94712223344', 'email': 'amara@example.com', 'room_number': '300', 'check_in': '2026-11-01', 'check_out': '2026-11-03', 'special_requests': None}}


In [31]:
# view by name
print(view_customer(name="Amara Perera"))

{'success': True, 'multiple_matches': False, 'result': {'customer_id': 1, 'full_name': 'Amara Perera', 'phone': '+94712223344', 'email': 'amara@example.com', 'room_number': '300', 'check_in': '2026-11-01', 'check_out': '2026-11-03', 'special_requests': None}}


In [32]:
# View, not found
print(view_customer(customer_id=99))

{'success': False, 'error': 'No matching customer found.'}


Update 

In [33]:
def update_customer(customer_id, full_name=None, phone=None, email=None,
                     room_number=None, check_in=None, check_out=None,
                     special_requests=None):
    existing = view_customer(customer_id=customer_id)
    if not existing["success"]:
        return {"success": False, "error": "Customer not found."}

    current = existing["result"]

    # Validate against the merged (existing + new) values so cross-field
    # rules like check_out > check_in still hold
    merged_check_in = check_in if check_in is not None else current["check_in"]
    merged_check_out = check_out if check_out is not None else current["check_out"]

    valid, error = validate_customer_data(
        full_name=full_name, phone=phone, email=email,
        room_number=room_number, check_in=merged_check_in, check_out=merged_check_out
    )
    if not valid:
        return {"success": False, "error": error}

    fields = {
        "full_name": full_name, "phone": phone, "email": email,
        "room_number": room_number, "check_in": check_in,
        "check_out": check_out, "special_requests": special_requests
    }
    changes = {k: v for k, v in fields.items() if v is not None}

    if not changes:
        return {"success": False, "error": "No fields provided to update."}

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        set_clause = ", ".join(f"{k} = ?" for k in changes)
        cursor.execute(
            f"UPDATE customers SET {set_clause} WHERE customer_id = ?",
            (*changes.values(), customer_id)
        )
        conn.commit()

    return {"success": True, "customer_id": customer_id, "changed_fields": changes}

In [15]:
# update customer data
print(update_customer(customer_id=1, room_number="300"))
# Expect: success True, changed_fields: {'room_number': '300'}
print(view_customer(customer_id=1))  # confirm the change actually persisted

{'success': True, 'customer_id': 1, 'changed_fields': {'room_number': '300'}}
{'success': True, 'multiple_matches': False, 'result': {'customer_id': 1, 'full_name': 'Amara Perera', 'phone': '+94712223344', 'email': 'amara@example.com', 'room_number': '300', 'check_in': '2026-11-01', 'check_out': '2026-11-03', 'special_requests': None}}


Delete

In [34]:
def delete_customer(customer_id):
    existing = view_customer(customer_id=customer_id)
    if not existing["success"]:
        return {"success": False, "error": "Customer not found."}

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM customers WHERE customer_id = ?", (customer_id,))
        conn.commit()

    return {"success": True, "customer_id": customer_id}

In [18]:
print(delete_customer(customer_id=3))
print(view_customer(customer_id=3))
# Expect: deleted successfully, then "No matching customer found."

{'success': False, 'error': 'Customer not found.'}
{'success': False, 'error': 'No matching customer found.'}


# Tools

In [35]:
save_customer_schema = {
    "type": "function",
    "function": {
        "name": "save_customer",
        "description": "Save a new customer record once all required details have been collected and confirmed with the staff member.",
        "parameters": {
            "type": "object",
            "properties": {
                "full_name": {"type": "string", "description": "Customer's full name"},
                "phone": {"type": "string", "description": "Phone number, 9-15 digits, may start with +"},
                "room_number": {"type": "string", "description": "Room number, e.g. 101 or A12"},
                "check_in": {"type": "string", "description": "Check-in date in YYYY-MM-DD format"},
                "check_out": {"type": "string", "description": "Check-out date in YYYY-MM-DD format"},
                "email": {"type": "string", "description": "Optional email address"},
                "special_requests": {"type": "string", "description": "Optional free-text special requests"}
            },
            "required": ["full_name", "phone", "room_number", "check_in", "check_out"],
            "additionalProperties": False
        }
    }
}

view_customer_schema = {
    "type": "function",
    "function": {
        "name": "view_customer",
        "description": "Look up an existing customer by their ID or by name.",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "integer", "description": "Exact customer ID to look up"},
                "name": {"type": "string", "description": "Full or partial name to search for"}
            },
            "additionalProperties": False
        }
    }
}

update_customer_schema = {
    "type": "function",
    "function": {
        "name": "update_customer",
        "description": "Update one or more fields of an existing customer. The customer_id itself can never be changed.",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "integer", "description": "ID of the customer to update"},
                "full_name": {"type": "string", "description": "New full name"},
                "phone": {"type": "string", "description": "New phone number"},
                "email": {"type": "string", "description": "New email address"},
                "room_number": {"type": "string", "description": "New room number"},
                "check_in": {"type": "string", "description": "New check-in date, YYYY-MM-DD"},
                "check_out": {"type": "string", "description": "New check-out date, YYYY-MM-DD"},
                "special_requests": {"type": "string", "description": "New special requests text"}
            },
            "required": ["customer_id"],
            "additionalProperties": False
        }
    }
}

delete_customer_schema = {
    "type": "function",
    "function": {
        "name": "delete_customer",
        "description": "Permanently delete a customer record. Only call this AFTER the staff member has explicitly confirmed with 'yes' in response to a confirmation question. If they say 'no', do not call this tool.",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "integer", "description": "ID of the customer to delete"}
            },
            "required": ["customer_id"],
            "additionalProperties": False
        }
    }
}



In [36]:
tools = [save_customer_schema, view_customer_schema, update_customer_schema, delete_customer_schema]

In [37]:
system_prompt = """You are a front-desk assistant for a hotel. You help staff add, view, update
and delete customer records by chatting naturally.

Rules you must always follow:
- Replies are polite, short and professional.
- When adding a new customer, ask for missing required details (full name, phone,
  room number, check-in date, check-out date) one or two at a time, not all at once.
  Email and special requests are optional - don't ask for them unless the staff member
  wants to add them.
- Never guess or invent a detail the staff member hasn't given you.
- Before deleting a customer, you must ask the staff member to confirm with a clear
  yes or no. Only call the delete_customer tool after they reply "yes". If they say
  "no", cancel the deletion and say so - do not call the tool.
- After any save, update or delete, clearly repeat back exactly what happened, so the
  staff member can check it.
- If a tool call fails (for example, invalid data or a customer not found), explain the
  problem in simple, friendly language and ask for a correction. Never say something
  succeeded if it did not.
- When looking up a customer, only show the records the staff member asked for.
- If several customers match a name search, list them briefly and ask which one is meant.
- If asked about anything unrelated to customer records (e.g. room pricing, payments,
  staff accounts), politely explain that you can only help with customer records.
"""

In [38]:
def handle_tool_call(tool_call):
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    if function_name == "save_customer":
        result = save_customer(**arguments)
    elif function_name == "view_customer":
        result = view_customer(**arguments)
    elif function_name == "update_customer":
        result = update_customer(**arguments)
    elif function_name == "delete_customer":
        result = delete_customer(**arguments)
    else:
        result = {"success": False, "error": f"Unknown tool: {function_name}"}

    return {
        "role": "tool",
        "content": json.dumps(result),
        "tool_call_id": tool_call.id
    }

In [39]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    finish_reason = response.choices[0].finish_reason

    while finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        tool_calls = message_obj.tool_calls

        messages.append(message_obj)
        for tool_call in tool_calls:
            tool_result = handle_tool_call(tool_call)
            messages.append(tool_result)

        response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason

    return response.choices[0].message.content

In [45]:
gradio.ChatInterface(fn=chat).launch(inbrowser=True, share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://0289d3993bd7d01bd6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "c:\Users\HP\Hotel_Management_AI_Agent\.venv\Lib\site-packages\gradio\queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\HP\Hotel_Management_AI_Agent\.venv\Lib\site-packages\gradio\route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "c:\Users\HP\Hotel_Management_AI_Agent\.venv\Lib\site-packages\gradio\blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "c:\Users\HP\Hotel_Management_AI_Agent\.venv\Lib\site-packages\gradio\blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\Hotel_M